# Introduction to MuJoCo

### What is MuJoCo


**MuJoCo** stands for **Mu**lti-**Jo**int dynamics with **Co**ntact. It is a general purpose physics engine that aims to facilitate research and development in robotics, biomechanics, graphics and animation, machine learning, and other areas that demand fast and accurate simulation of articulated structures interacting with their environment.

Complete [documentation](https://mujoco.readthedocs.io/en/stable/APIreference/index.html).

Original [paper](https://ieeexplore.ieee.org/document/6386109).


#### Key features

- Simulation in generalized coordinates
- Optimization-based contact dynamics
- Interactive simulation and visualization
- Separation of model and data

## MuJoCo Basics

We begin by defining and loading a simple model:

In [1]:
import mujoco

pendulum = """
<mujoco>

  <worldbody>
    <body name="pendulum" euler="0 0 90">
      <geom name="blue_rod" type="cylinder" pos="0. 0. .4" size=".025 .35" euler="0 0 0" rgba="0 0 1 1"/>
      <geom name="green_sphere" pos=".0 .0 .0" size=".1" rgba="0 1 0 1"/>
      <geom name="red_box" type="box" pos=".0 .0 .7" size=".08 .08 .08" rgba="1 0 0 1"/>
    </body>
  </worldbody>
  
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(pendulum)

The `xml` string is written in MuJoCo's [MJCF](http://www.mujoco.org/book/modeling.html), which is an [XML](https://en.wikipedia.org/wiki/XML#Key_terminology)-based modeling language.
  - The only required element is `<mujoco>`. The smallest valid MJCF model is `<mujoco/>` which is a completely empty model.
  - All physical elements live inside the `<worldbody>` which is always the top-level body and constitutes the global origin in Cartesian coordinates.
  - We define a `body` pendulum with three geoms in the world named `red_box`, `green_sphere` and `blue_rod`. In `body` one can specify the mass and inertia of the object while the `geom` specify the visual and the collision model of the parent `body`. 

The MJCF language is described in the documentation's [XML Reference chapter](https://mujoco.readthedocs.io/en/latest/XMLreference.html).

The `from_xml_string()` method invokes the model compiler, which creates a binary `mjModel` instance. One can also use `from_xml_path()` to load a model from a file.

### `MjModel`

MuJoCo's `mjModel`, contains the *model description*, i.e., all quantities which **do not change over time**.

The complete description of `mjModel` can be found at the end of the header file [`mjmodel.h`](https://github.com/google-deepmind/mujoco/blob/259d3876ce6816f8572330118487f5f80c4295fa/include/mujoco/mjmodel.h#L565). Note that the header files contain short, useful inline comments, describing each field.

Examples of quantities that can be found in `mjModel` are:
 - `ngeom`: number of geoms in the scene
 - `nbody`: number of bodies in the scene
 - `geom_rgba`: colors of the geometries
 - `nq`: number of generalized coordinates
 - `nv`: number of DoF
 - `nu`: number of actuators
 - `opt`: physics option


In [2]:
model.ngeom

3

In [3]:
model.geom_rgba

array([[0., 0., 1., 1.],
       [0., 1., 0., 1.],
       [1., 0., 0., 1.]], dtype=float32)

**Question:**
- What is the number of degree of freedom of our model?
- What is the intergration timestep (see the [documentation](https://mujoco.readthedocs.io/en/stable/APIreference/APItypes.html#mjoption))?

In [4]:
print("Number of DoF",model.nv)
print ("Integration timestep", model.opt.timestep)



Number of DoF 0
Integration timestep 0.002


As you can see our model as $0$ [degrees of freedom](https://www.google.com/url?sa=D&q=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FDegrees_of_freedom_(mechanics)), why is that?

We add DoFs by adding *joints* to bodies (things that move and which have inertia are called *bodies*), specifying how they can move with respect to their parents. Let's make a new body that contains our *geoms*, add a hinge joint.

Let's also define a start position that is not at equilibrium with a `<keyframe>`.

In [5]:
pendulum = """
<mujoco>

  <worldbody>
    <light name="top" pos="0 0 2"/>
    <body name="pendulum" euler="0 0 90">
      <joint name="swing" type="hinge" axis="1 0 0" pos="0. 0. .5"/>
      <geom name="blue_rod" type="cylinder" pos="0. 0. .4" size=".025 .35" euler="0 0 0" rgba="0 0 1 1"/>
      <geom name="green_sphere" pos=".0 .0 .0" size=".1" rgba="0 1 0 1"/>
      <geom name="red_box" type="box" pos=".0 .0 .7" size=".08 .08 .08" rgba="1 0 0 1"/>
    </body>
  </worldbody>
  
  <keyframe>
    <key name="home" qpos="0.3"/>
  </keyframe>
  
</mujoco>
"""


## Named access

The MuJoCo Python bindings provide convenient [accessors](https://mujoco.readthedocs.io/en/latest/python.html#named-access) using names.

Calling the named accessor without specifying a property will tell us what all the valid properties are:

In [6]:
model.geom('green_sphere')

<_MjModelGeomViews
  bodyid: array([1], dtype=int32)
  conaffinity: array([1], dtype=int32)
  condim: array([3], dtype=int32)
  contype: array([1], dtype=int32)
  dataid: array([-1], dtype=int32)
  friction: array([1.e+00, 5.e-03, 1.e-04])
  gap: array([0.])
  group: array([0], dtype=int32)
  id: 1
  margin: array([0.])
  matid: array([-1], dtype=int32)
  name: 'green_sphere'
  pos: array([0., 0., 0.])
  priority: array([0], dtype=int32)
  quat: array([1., 0., 0., 0.])
  rbound: array([0.1])
  rgba: array([0., 1., 0., 1.], dtype=float32)
  sameframe: array([1], dtype=uint8)
  size: array([0.1, 0. , 0. ])
  solimp: array([9.0e-01, 9.5e-01, 1.0e-03, 5.0e-01, 2.0e+00])
  solmix: array([1.])
  solref: array([0.02, 1.  ])
  type: array([2], dtype=int32)
  user: array([], dtype=float64)
>

Let's read the `green_sphere`'s rgba values:

In [7]:
model.geom('green_sphere').rgba

array([0., 1., 0., 1.], dtype=float32)

Calling the `model.geom()` accessor without a name string generates a convenient error that tells us what the valid names are.

In [8]:
try:
  model.geom()
except KeyError as e:
  print(e)

"Invalid name ''. Valid names: ['blue_rod', 'green_sphere', 'red_box']"


This functionality is a convenience shortcut for MuJoCo's [`mj_name2id`](https://mujoco.readthedocs.io/en/latest/APIreference.html?highlight=mj_name2id#mj-name2id) function:

In [9]:
id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_GEOM, 'green_sphere')
model.geom_rgba[id, :]

array([0., 1., 0., 1.], dtype=float32)

Bodies and gemetries have specific and potentially different ids. Note that the 0th body is always the `world`. It cannot be renamed.

The `id` and `name` attributes are useful in Python comprehensions:

In [10]:
def print_id_geom_name(model):
    for i in range(model.ngeom):
        print("geom id:", i, "geom name", model.geom(i).name)

def print_id_body_name(model):
    for i in range(model.nbody):
        print("body id:", i, "body name", model.body(i).name)


print_id_geom_name(model)
print_id_body_name(model)

geom id: 0 geom name blue_rod
geom id: 1 geom name green_sphere
geom id: 2 geom name red_box
body id: 0 body name world
body id: 1 body name pendulum


**Question:**
- What is the friction coefficient of the `red_box`?
- Set the transparency of the `blue_rod` to `0.5`?

In [11]:
model.geom('blue_rod').rgba[3] = 0.5

In [12]:
print("friction coefficient",model.geom('red_box').friction)


friction coefficient [1.e+00 5.e-03 1.e-04]


### `MjData`

`mjData` contains the **state** and **quantities that depend on it**. The state is made up of time, [generalized](https://en.wikipedia.org/wiki/Generalized_coordinates) positions and generalized velocities. These are respectively `data.time`, `data.qpos` and `data.qvel`. What can be accessed in the `mjData` is described in the [documentation](https://mujoco.readthedocs.io/en/stable/APIreference/APItypes.html#mjdata). In order to make a new `mjData`, all we need is our `mjModel`

In [13]:
data = mujoco.MjData(model)

`mjData` also contains **functions of the state**, for example the Cartesian positions of objects in the world frame. The (x, y, z) positions of our two geoms are in `data.geom_xpos`:

Examples of quantities that can be found in `mjData` are:
 - `qpos`: position (nq x 1)
 - `qvel`: velocity (nv x 1)
 - `qacc`: acceleration (nv x 1)
 - `ctrl`: actuator control (nu x 1)
 - `xpos`: cartesian position of the body frame (nbody x 3)
 - `xipos`: cartesian position of the body com (nbody x 3)
 - `xquat`: cartesian orientation of the body frame (nbody x 4)
 - `xmat`: cartesian orientation of the body frame (nbody x 9)


Quantities in `mjData` need to be explicitly propagated (see [below](#scrollTo=QY1gpms1HXeN)). In our case, the minimal required function is [`mj_kinematics`](https://mujoco.readthedocs.io/en/latest/APIreference.html#mj-kinematics), which computes global Cartesian poses for all objects (excluding cameras and lights).

In [14]:
print('Init:\n', data.geom_xpos)
mujoco.mj_kinematics(model, data)
print('\nUpated:\n', data.geom_xpos)

# MjData also supports named access:
print('\nnamed access:\n', data.geom('red_box').xpos)

Init:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]

Upated:
 [[0.  0.  0.4]
 [0.  0.  0. ]
 [0.  0.  0.7]]

named access:
 [0.  0.  0.7]


**Questions**:
- What is the current orientation of the `red_box` (as a rotation matrix)?
- What is the current angle of the 1D pendulum around it's axis?

In [15]:
print ('Current Orientation of red_box\n', data.geom('red_box').xmat)

model = mujoco.MjModel.from_xml_string(pendulum) 
data = mujoco.MjData(model)
angle = data.qpos[0]

print ("Current Angle of the pendulum", angle)

Current Orientation of red_box
 [ 2.22044605e-16 -1.00000000e+00  0.00000000e+00  1.00000000e+00
  2.22044605e-16  0.00000000e+00  0.00000000e+00  0.00000000e+00
  1.00000000e+00]
Current Angle of the pendulum 0.0


## Simulation

An interactive GUI viewer is provided as part of the Python package in the `mujoco.viewer` module. There are several ways to launch the viewer.

One of them is to call `viewer.launch_passive(model, data)`. This function does not block, allowing user code to continue execution. In this mode, the user’s script is responsible for timing and advancing the physics state, and mouse-drag perturbations will not work unless the user explicitly synchronizes incoming events.

Alternatevely, one can record frames each simulation step and replay them but this is less convenient as the viewer provides some interesting visualization tools. See this [tutorial](https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb) for example.

### Basics

At each step in the simulation loop:
- Compute the forward dynamics: $\dot{x}_{t+h} = f(x_t)$, where $x$ is the state and $h$ the timestep. This is done with `mujoco.mj_step(model, data)`
- Update the viewer with `viewer.sync()`
- Eventually pause the simulation loop for a "real time" simulation
- One can also add callbacks and change the viewer options. The [documention](https://mujoco.readthedocs.io/en/stable/python.html#passive-viewer) provides exemples.

Let's run the simulation of this model in the build-in viewer.

**Questions:**
- Complete the simulation loop in the function `sim_viewer`.

In [16]:
import time
import mujoco.viewer
from utils import save_video_from_frames

def run_simulation(model, data, sim_time=5, use_viewer=True):
	# With interactive viewer
	if use_viewer:
		viewer = mujoco.viewer.launch_passive(model, data)
		# Visualize joints
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True
	
	# Use renderer without display and save video
	else:
		H, W = 480, 640
		FPS = 30
		renderer = mujoco.Renderer(model, H, W)
		frames = []

	### Simulation loop
	while (
		(not use_viewer or viewer.is_running()) and
		data.time < sim_time
	):
		step_start = time.time()
		
		# Compute forwards dynamics
		mujoco.mj_step (model,data)

		# Pick up changes to the physics state, apply perturbations, update options from GUI.
		# TODO

		# Render new state in the viewer
		if use_viewer:
			viewer.sync() # TODO

		# Save frame at FPS rate
		elif len(frames) < data.time * FPS:
			renderer.update_scene(data)
			pixels = renderer.render()
			frames.append(pixels)

		# Rudimentary time keeping, will drift relative to wall clock.
		time_until_next_step = model.opt.timestep - (time.time() - step_start)
		if time_until_next_step > 0:
			time.sleep(time_until_next_step)
	
	if use_viewer:
		if viewer.is_running():
			viewer.close()
	else:
		FILENAME = "simulation.mp4"
		save_video_from_frames(FILENAME, frames, FPS)


In [23]:
model = mujoco.MjModel.from_xml_string(pendulum)
data = mujoco.MjData(model)

# Start the pendulum from the keyframe
mujoco.mj_resetDataKeyframe(model, data, 0)
run_simulation(model, data)

The visualizer is interactive which allows the user to interact with the different bodies.
To do so, double click on the body (should be highlighted), then press *alt+cltr* **and** *drag simulatneously the mouse* to exerce an external translational force (press only alt force a rotational force).

The viewer also provides information about all the physics parameters that can be changed interactively. Look at the `Physics` tab.

**Question:**

- Try for instance to change the integration step time or to reverse the gravity. This will change the arguments of the model class, so you might need to reinstantiate it to reverse your change.


In [22]:
run_simulation(model, data)

In [37]:
import numpy as np
model.opt.gravity = np.array([0,0,9.81])

In [42]:
# Reset model
model = mujoco.MjModel.from_xml_string(pendulum)
data = mujoco.MjData(model)
print('default gravity', model.opt.gravity)

# Flip gravity
model.opt.gravity = (0, 0, 10)
print('flipped gravity', model.opt.gravity)

# Change optimization step
model.opt.timestep = 0.001

run_simulation(model, data)

default gravity [ 0.    0.   -9.81]
flipped gravity [ 0.  0. 10.]


This can also be done by changing the model parameters (see the [documentation](https://mujoco.readthedocs.io/en/stable/APIreference/APItypes.html#mjoption)).

In [43]:
# Reset model
model = mujoco.MjModel.from_xml_string(pendulum)
data = mujoco.MjData(model)
print('default gravity', model.opt.gravity)

# Flip gravity
model.opt.gravity = (0, 0, 10)
print('flipped gravity', model.opt.gravity)

# Change optimization step
model.opt.timestep = 0.001

run_simulation(model, data)

default gravity [ 0.    0.   -9.81]
flipped gravity [ 0.  0. 10.]


or directely in the model description file.
```xml
<mujoco>
  <option gravity="0 0 10" integrator="RK4" timestep="0.001"/>
  ...
</mujoco>
```

Here is a more advance example with a chaotic pendulum

In [44]:
chaotic_pendulum = """
<mujoco>
  <option timestep=".001">
    <flag energy="enable" contact="disable"/>
  </option>

  <default>
    <joint type="hinge" axis="0 -1 0"/>
    <geom type="capsule" size=".02"/>
  </default>

  <worldbody>
    <light pos="0 -.4 1"/>
    <camera name="fixed" pos="0 -1 0" xyaxes="1 0 0 0 0 1"/>
    <body name="0" pos="0 0 .2">
      <joint name="root"/>
      <geom fromto="-.2 0 0 .2 0 0" rgba="1 1 0 1"/>
      <geom fromto="0 0 0 0 0 -.25" rgba="1 1 0 1"/>
      <body name="1" pos="-.2 0 0">
        <joint/>
        <geom fromto="0 0 0 0 0 -.2" rgba="1 0 0 1"/>
      </body>
      <body name="2" pos=".2 0 0">
        <joint/>
        <geom fromto="0 0 0 0 0 -.2" rgba="0 1 0 1"/>
      </body>
      <body name="3" pos="0 0 -.25">
        <joint/>
        <geom fromto="0 0 0 0 0 -.2" rgba="0 0 1 1"/>
      </body>
    </body>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(chaotic_pendulum)
data = mujoco.MjData(model)

# set initial velocity
mujoco.mj_resetData(model, data)
data.joint('root').qvel = 10

run_simulation(model, data)

### MjSpec

Rewritting the `.xml` in order to change the model can be unconvinent.
What if I want to change the location of some objects in the scene? Will I have to change manually the position in the `.xml` all the times?

The answer is **no**, as MuJoCo recently provided a feature called [`MjSpec`](https://mujoco.readthedocs.io/en/stable/python.html#model-editing) that helps directly create and edit the model from code lines without the need of an `.xml` file.
This feature is a bit too advanced for this introduction tutorial.
Below is an example on how to build the same simple pendulum seen before using `MjSpec`.

In [45]:
spec = mujoco.MjSpec()
spec.modelname = "pendulum_spec"
spec.worldbody.add_light(name="top", pos=[0, 0, 2])

# Add pendulum body
body = spec.worldbody.add_body(name="pendulum", euler=[0, 0, 90])
body.add_joint(
    name="swing",
    type=mujoco.mjtJoint.mjJNT_HINGE,
    axis=[1, 0, 0],
    pos=[0, 0, 0.5]
)

# Geoms
body.add_geom(
    name="blue_rod",
    type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    pos=[0, 0, 0.4],
    size=[0.025, 0.35, 0.],
    rgba=[0, 0, 1, 1]
)
body.add_geom(
    name="green_sphere",
    type=mujoco.mjtGeom.mjGEOM_SPHERE,
    pos=[0, 0, 0],
    size=[0.1, 0, 0],
    rgba=[0, 1, 0, 1]
)
body.add_geom(
    name="red_box",
    type=mujoco.mjtGeom.mjGEOM_BOX,
    pos=[0, 0, 0.7],
    size=[0.08, 0.08, 0.08],
    rgba=[1, 0, 0, 1]
)

# Add keyframe
spec.add_key(name="home", qpos=[0.3])

# spec can be edited until it's compiled into a model.
# Compile to actual model
model = spec.compile()
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, 0)

run_simulation(model, data)

## Control

We've seen so far how to define a model, run the simulation and render it in the viewer.
MuJoCo can also model actuators (such as cylinders and biological muscles) that have their own activation states assembled in the vector `mjData.act`.

#### Basics

In this section we will see how to define a control law for the simple pendulum we've seen at the beginning. Or goal is to make it stand still horizontally. We add a mass to the pendulum. Let's define a simple PD controller for this.

In [46]:
pendulum = """
<mujoco>

  <worldbody>
    <light name="top" pos="0 0 2"/>
    <body name="pendulum" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 0 0" pos="0. 0. .5"/>
      <geom name="blue_rod" type="cylinder" pos="0. 0. .4" size=".025 .35" euler="0 0 0" rgba="0 0 1 1"/>
      <geom name="green_sphere" pos=".0 .0 .0" size=".1" rgba="0 1 0 1"/>
      <geom name="red_box" type="box" pos=".0 .0 .7" size=".08 .08 .08" rgba="1 0 0 1"/>
    </body>
  </worldbody>

  <keyframe>
    <key name="home" qpos="0.3"/>
  </keyframe>
  
</mujoco>
"""

In [47]:
import numpy as np
from typing import Union

class PDController():
    def __init__(self,
                 kp: Union[float, np.ndarray] = 1.,
                 kd: Union[float, np.ndarray] = 1.
                 ) -> None:
      self.kp = kp
      self.kd = kd
      
      self.q_des = 0.
      
    def set_command(self, q_des: np.ndarray) -> None:
        self.q_des = q_des
      
    def get_torques(self, q: np.ndarray, v: np.ndarray) -> np.ndarray:
        torques = self.kp * (q - self.q_des) + self.kd * (np.zeros_like(v) - v)
        return torques

controller = PDController()

We also need to adapt the simulation loop to change the command to the joint at each simulation step. This is done by setting `data.ctrl` to the desired command.
We also add a callback to **trigger the controller by pressing the spacebar**.

In [ ]:
def run_simulation(model, data, controller=None, use_viewer=True, sim_time=5):
	use_controller = True
	if use_viewer:
		# Use key callback for the viewer
		def key_callback(keycode):
			if chr(keycode) == ' ':
				nonlocal use_controller
				use_controller = not use_controller
				
		viewer = mujoco.viewer.launch_passive(model, data, key_callback=key_callback)
		# Visualize joints
		viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

	else:
		H, W = 480, 640
		FPS = 30
		renderer = mujoco.Renderer(model, H, W)
		frames = []

	### Simulation loop
	while (
		(not use_viewer or viewer.is_running()) and
		data.time < sim_time
	):
		step_start = time.time()

		# Control loop
		if controller != None:
			torque = np.zeros(model.nv)
			if use_controller:
				q, v = data.qpos, data.qvel
				# TODO: get the torques from the controller
				torque = controller.get_torques(q,v)

			# TODO: update the model data with the computed torques
			data.crtl = torque

		# Render new state in the viewer
		if use_viewer:
			viewer.sync()

		# Save frame at FPS rate
		elif len(frames) < data.time * FPS:
			renderer.update_scene(data)
			pixels = renderer.render()
			frames.append(pixels)

		# Compute forwards dynamics
		mujoco.mj_step(model, data)

		# Rudimentary time keeping, will drift relative to wall clock.
		time_until_next_step = model.opt.timestep - (time.time() - step_start)
		if time_until_next_step > 0:
			time.sleep(time_until_next_step)
	
	if use_viewer:
		if viewer.is_running():
			viewer.close()
	else:
		FILENAME = "simulation.mp4"
		print(frames[0].shape)
		save_video_from_frames(FILENAME, frames, FPS)

model = mujoco.MjModel.from_xml_string(pendulum)
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, 0)

Q_DES = np.pi/2 # rad
controller.set_command(Q_DES)
run_simulation(model, data, controller)

Nothing happened... this is because one need to add an actuator in the model MJCF description with the flag `<actuator>`.
Let's try again...

In [50]:
pendulum_ctrl = """
<mujoco>

  <worldbody>
    <light name="top" pos="0 0 2"/>
    <body name="pendulum" euler="0 0 -30">
      <joint name="swing" type="hinge" axis="1 0 0" pos="0. 0. .5"/>
      <geom name="blue_rod" type="cylinder" pos="0. 0. .4" size=".025 .35" euler="0 0 0" rgba="0 0 1 1"/>
      <geom name="green_sphere" pos=".0 .0 .0" size=".1" rgba="0 1 0 1"/>
      <geom name="red_box" type="box" pos=".0 .0 .7" size=".08 .08 .08" rgba="1 0 0 1"/>
    </body>
  </worldbody>
  
  <actuator>
    <motor name="my_motor" joint="swing" gear="1"/>
  </actuator>

  <keyframe>
    <key name="home" qpos="0.3"/>
  </keyframe>
  
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(pendulum_ctrl)
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, 0)

# For torque control
model.actuator_gainprm[:, 0] = 1

KP = -50
KD = 20
controller = PDController(KP, KD)
controller.set_command(Q_DES)

run_simulation(model, data, controller)

**Question (only if you have some time):**
- Define a model of a double pendulum and use a PD controller to make it stand horizontally.